<a href="https://colab.research.google.com/github/guitorte/audio/blob/claude/stem-midi-converter-OwEoi/stem-to-midi/notebooks/Stem_to_MIDI_Batch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stem → MIDI — Batch (multi-track)

Converte **todos os stems de uma faixa Demucs** em um único arquivo MIDI multi-track.

Engines escolhidas automaticamente por stem (mesmo dispatcher do MVP):

| Stem | Engine | GM program |
|---|---|---|
| `vocals` | Basic Pitch (ONNX) | 53 — Voice Oohs |
| `bass`   | Basic Pitch (ONNX) | 33 — Electric Bass |
| `guitar` | Basic Pitch (ONNX) | 27 — Electric Guitar (clean) |
| `piano`  | Basic Pitch (ONNX) |  0 — Acoustic Grand Piano |
| `other`  | Basic Pitch (ONNX) |  0 — Acoustic Grand Piano |
| `drums`  | ADTOF-pytorch      | canal 10 (GM drum map) |

## Convenção de paths no Drive

| | Path |
|---|---|
| Entrada (stems Demucs) | `/content/drive/MyDrive/stem-to-midi/input/<track>/` |
| Saída (per-stem + merged) | `/content/drive/MyDrive/stem-to-midi/output/<track>/` |

A pasta de entrada deve conter os arquivos do Demucs nomeados pelos stem types: `vocals.wav`, `drums.wav`, `bass.wav`, `guitar.wav`, `piano.wav`, `other.wav` (todos opcionais — o que estiver lá é transcrito). Saidas do Demucs `htdemucs_6s` já têm esse layout.

## 1. Instalar dependências

⚠️ Após rodar esta célula, **reinicie o runtime** (`Runtime ▸ Restart Session`) e siga das células seguintes.

In [ ]:
# Colab roda Python 3.12. basic-pitch 0.4.0 puxa tensorflow<2.15.1 /
# tflite-runtime, que não têm wheel 3.12. Workaround: instalar
# basic-pitch --no-deps e cair no backend ONNX (modelo nmp.onnx já vem
# embutido). Ver spotify/basic-pitch#188.
#
# ADTOF-pytorch é PyTorch-only e tem pesos embutidos.

!pip uninstall -y basic-pitch tensorflow tflite-runtime 2>/dev/null
!pip install basic-pitch --no-deps onnxruntime
!pip install "resampy<0.4.3" librosa pretty_midi mir_eval scikit-learn scipy typing_extensions soundfile mido matplotlib flatbuffers protobuf
!pip install git+https://github.com/xavriley/ADTOF-pytorch.git

import importlib.util
_required = ('basic_pitch', 'onnxruntime', 'adtof_pytorch', 'pretty_midi', 'librosa', 'soundfile', 'mido')
_missing = [m for m in _required if importlib.util.find_spec(m) is None]
print('=' * 60)
if _missing:
    print(f'Módulos não encontrados após install: {_missing}')
    print('   A célula 2 tentará reinstalar automaticamente.')
else:
    print('Instalação concluída — Basic Pitch (ONNX) + ADTOF-pytorch OK.')
print('Reinicie o runtime (Runtime > Restart Session) e siga adiante.')
print('=' * 60)

## 2. Imports e clone do repositório

Esta célula busca os módulos `stem-to-midi/modules/` (onde mora `batch_stems_to_midi`). Em Colab fazemos um shallow clone do repositório para `/content/audio` para usar o pacote sem instalar.

In [ ]:
import os, sys, subprocess, importlib, warnings
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

def _pip(*args):
    print('pip', *args)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *args])

def _ensure(module_name, *, pip_args=None):
    try:
        return importlib.import_module(module_name)
    except ImportError:
        _pip(*(pip_args or [module_name.replace('_', '-')]))
        importlib.invalidate_caches()
        return importlib.import_module(module_name)

# Pitched: backend ONNX (não dependemos de TF/TFLite — quebram em Py 3.12)
onnxruntime = _ensure('onnxruntime')
try:
    import basic_pitch.inference  # noqa: F401
except ImportError:
    _pip('basic-pitch', '--no-deps')
    _pip('resampy<0.4.3', 'librosa', 'pretty_midi', 'mir_eval',
         'scikit-learn', 'scipy', 'typing_extensions', 'flatbuffers', 'protobuf')
    importlib.invalidate_caches()
    import basic_pitch.inference  # noqa: F401

# Drums
try:
    import adtof_pytorch  # noqa: F401
except ImportError:
    _pip('git+https://github.com/xavriley/ADTOF-pytorch.git')
    importlib.invalidate_caches()
    import adtof_pytorch  # noqa: F401

librosa     = _ensure('librosa')
_ensure('librosa.display')
pretty_midi = _ensure('pretty_midi', pip_args=['pretty_midi'])
soundfile   = _ensure('soundfile')
mido        = _ensure('mido')

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Pega os módulos `stem-to-midi/modules/` deste repositório.
REPO_DIR = '/content/audio' if IN_COLAB else os.path.dirname(os.path.dirname(os.path.abspath(os.getcwd())))
if IN_COLAB and not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--depth', '1',
                           '--branch', 'claude/stem-midi-converter-OwEoi',
                           'https://github.com/guitorte/audio.git', REPO_DIR])
PKG_DIR = os.path.join(REPO_DIR, 'stem-to-midi')
if PKG_DIR not in sys.path:
    sys.path.insert(0, PKG_DIR)

from modules import batch_stems_to_midi, STEM_PRESETS  # noqa: E402
from modules.transcribe import DEFAULT_STEM_PROGRAMS  # noqa: E402
from IPython.display import Audio, display, FileLink  # noqa: E402

print(f'Colab           : {IN_COLAB}')
print(f'ONNX runtime    : {onnxruntime.__version__}')
print(f'Engines prontas : Basic Pitch (pitched) + ADTOF-pytorch (drums)')
print(f'Stem types reconhecidos: {sorted(STEM_PRESETS.keys())}')

## 3. Montar Google Drive

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print('Drive montado.')
else:
    print('Fora do Colab — pulando mount.')

## 4. Configurar paths

Aponte `STEMS_DIR` para uma pasta com os stems do Demucs. A saída do `htdemucs_6s` já tem o layout esperado (arquivos `vocals.wav`, `drums.wav`, `bass.wav`, `guitar.wav`, `piano.wav`, `other.wav`).

Se algum arquivo tiver nome diferente, use o mapping `STEM_RENAMES` abaixo para forcá-lo a um tipo conhecido.

In [ ]:
# @title Parâmetros
BASE_DIR    = '/content/drive/MyDrive/stem-to-midi'  # @param {type:'string'}
TRACK_NAME  = 'minha-faixa'                          # @param {type:'string'}
STEMS_DIR   = ''                                     # @param {type:'string'}

# Mapeamento opcional para arquivos com nomes não-padrão.
# Ex.: {'lead_vox': 'vocals', 'kit_drums': 'drums'}
STEM_RENAMES = {}

if not STEMS_DIR:
    STEMS_DIR = os.path.join(BASE_DIR, 'input', TRACK_NAME)
OUTPUT_DIR  = os.path.join(BASE_DIR, 'output', TRACK_NAME)
MERGED_PATH = os.path.join(OUTPUT_DIR, f'{TRACK_NAME}.mid')

os.makedirs(STEMS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Stems dir   : {STEMS_DIR}')
print(f'Output dir  : {OUTPUT_DIR}')
print(f'Merged MIDI : {MERGED_PATH}')

if os.path.isdir(STEMS_DIR):
    files = sorted(f for f in os.listdir(STEMS_DIR)
                   if os.path.splitext(f)[1].lower() in ('.wav', '.mp3', '.flac', '.m4a', '.ogg'))
    print(f'\nArquivos de audio em STEMS_DIR ({len(files)}):')
    for f in files:
        full = os.path.join(STEMS_DIR, f)
        size_mb = os.path.getsize(full) / 1024**2
        print(f'  {f:30s}  {size_mb:6.2f} MB')
    if not files:
        print(f'  (vazio — coloque os stems do Demucs em {STEMS_DIR}/)')
else:
    print(f'\nSTEMS_DIR não existe. Crie-o e coloque os stems lá.')

## 5. Batch → multi-track MIDI

Cada stem reconhecido vira um arquivo `<stem_type>.mid` em `OUTPUT_DIR`, e todos são consolidados em `MERGED_PATH`. Stems com nome não-reconhecido são ignorados — use `STEM_RENAMES` para remapear.

In [ ]:
import time
_t0 = time.time()
result = batch_stems_to_midi(
    stems_dir=STEMS_DIR,
    output_path=MERGED_PATH,
    stem_types=STEM_RENAMES or None,
)
_dt = time.time() - _t0

print(f'\n=== Resumo ===')
print(f'Tempo total       : {_dt:.1f}s')
print(f'Stems processados : {len(result.stem_results)}')
print(f'Total de notas    : {result.total_notes}')
print(f'Duração do MIDI    : {result.duration_s:.1f}s')
print(f'MIDI consolidado  : {result.midi_path}')
print()
print(f'{"Stem":10s} {"Engine":14s} {"Notas":>6s} {"Duração":>10s}')
print('-' * 44)
for stem_type, r in result.stem_results.items():
    print(f'{stem_type:10s} {r.engine:14s} {r.n_notes:6d} {r.duration_s:9.1f}s')

## 6. Piano roll multi-track

Cada track aparece em uma cor diferente. Drums (canal 10) ficam em uma faixa separada com rótulos GM.

In [ ]:
merged = result.midi_data
if not merged.instruments:
    print('Nenhuma faixa no MIDI.')
else:
    NOTE = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
    GM_DRUMS_SHORT = {35:'BD', 36:'Kick', 38:'Snare', 39:'Clap', 40:'ElSnr',
                      41:'LoFlT', 42:'HH', 43:'HiFlT', 44:'PdHH', 45:'LoT',
                      46:'OpHH', 47:'LMT', 48:'HMT', 49:'Crash', 50:'HiT', 51:'Ride'}
    COLORS = {'vocals':'#d62728', 'bass':'#1f77b4', 'guitar':'#2ca02c',
              'piano':'#ff7f0e', 'other':'#9467bd', 'drums':'#7f7f7f'}

    pitched = [i for i in merged.instruments if not i.is_drum and i.notes]
    drums   = [i for i in merged.instruments if i.is_drum and i.notes]
    n_rows  = (1 if pitched else 0) + (1 if drums else 0)
    if n_rows == 0:
        print('Nenhuma nota nos stems.')
    else:
        fig, axes = plt.subplots(n_rows, 1, figsize=(14, 4*n_rows), squeeze=False)
        ax_idx = 0

        if pitched:
            ax = axes[ax_idx, 0]; ax_idx += 1
            for inst in pitched:
                color = COLORS.get(inst.name, '#000000')
                for n in inst.notes:
                    ax.barh(n.pitch, n.end - n.start, left=n.start, height=0.9,
                            color=color, alpha=0.65, edgecolor='none')
                ax.scatter([], [], color=color, label=f'{inst.name} (n={len(inst.notes)})')
            ax.set_xlabel('Tempo (s)')
            ax.set_ylabel('Pitch MIDI')
            ax.set_title('Pitched tracks', fontweight='bold')
            ax.legend(loc='upper right', framealpha=0.9)
            ax.grid(True, axis='y', alpha=0.3)

        if drums:
            ax = axes[ax_idx, 0]
            for inst in drums:
                for n in inst.notes:
                    ax.barh(n.pitch, max(0.05, n.end - n.start), left=n.start, height=0.85,
                            color=COLORS['drums'], alpha=0.8, edgecolor='none')
            unique_pitches = sorted({n.pitch for inst in drums for n in inst.notes})
            ax.set_yticks(unique_pitches)
            ax.set_yticklabels([GM_DRUMS_SHORT.get(p, f'p{p}') for p in unique_pitches])
            ax.set_xlabel('Tempo (s)')
            ax.set_title('Drum track (GM channel 10)', fontweight='bold')
            ax.grid(True, axis='y', alpha=0.3)

        plt.tight_layout()
        plt.show()

## 7. Preview sonoro do MIDI consolidado

Pitched tracks: senoidal via `pretty_midi.Instrument.synthesize`. Drums: noise/tone bursts por hit (igual ao MVP). Tudo somado e normalizado.

In [ ]:
sr_preview = 22050
dur_total = max(result.duration_s, 0.5)
wave = np.zeros(int(dur_total * sr_preview) + sr_preview, dtype=np.float32)

for inst in merged.instruments:
    if not inst.notes:
        continue
    if inst.is_drum:
        for nt in inst.notes:
            idx = int(nt.start * sr_preview)
            length = int(0.08 * sr_preview)
            env = np.exp(-np.linspace(0, 6, length))
            t = np.arange(length) / sr_preview
            if nt.pitch in (35, 36):
                tone = np.sin(2*np.pi*60*t)
            elif nt.pitch in (38, 40):
                tone = np.random.randn(length) * 0.5 + 0.3 * np.sin(2*np.pi*200*t)
            else:
                tone = np.random.randn(length) * 0.8
            sl = wave[idx:idx+length]
            sl[:] = sl + (tone * env * (nt.velocity/127.0)).astype(np.float32)[:len(sl)]
    else:
        chunk = inst.synthesize(fs=sr_preview)
        if len(chunk) > len(wave):
            chunk = chunk[:len(wave)]
        wave[:len(chunk)] += chunk.astype(np.float32)

peak = float(np.max(np.abs(wave)))
if peak > 0:
    wave = (0.9 * wave / peak).astype(np.float32)

print('MIDI consolidado (preview sintetizado):')
display(Audio(wave, rate=sr_preview))

## 8. Download do MIDI multi-track

O arquivo consolidado fica em `MERGED_PATH`. Os per-stem `.mid` ficam em `OUTPUT_DIR/<stem>.mid` — úteis para edição isolada em DAW.

In [ ]:
print(f'Arquivo consolidado: {MERGED_PATH}')
display(FileLink(MERGED_PATH))
print('\nPer-stem .mid files:')
for stem_type in result.stem_results:
    p = os.path.join(OUTPUT_DIR, f'{stem_type}.mid')
    if os.path.exists(p):
        print(f'  {stem_type:8s} -> {p}')
        display(FileLink(p))

if IN_COLAB:
    try:
        from google.colab import files
        files.download(MERGED_PATH)
    except Exception as e:
        print(f'(download manual via Drive — botão automático falhou: {e})')